# **Cosmological inference from the BAO-fit results for mocks**

This notebook shows how to run cosmological inference under $\Lambda$CDM from the BAO-fit results on mocks (vaying $hr_{\rm d}$ and $\Omega_{\rm m}$).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
import pandas as pd
from utils_data import GetThetaLimits
from utils_baofit import BAOFitInitializer
from utils_inference import BAOFitChecker, BAOInference
%matplotlib inline

# # Option A. DESY6-related stuff
# dataset_mocks_list = ["DESY6_COLA"]
# # dataset_mocks_list = ["DESY6_COLA_DR1tiles_noDESI"]
# # dataset_mocks_list = ["DESY6_COLA_DR1tiles_DESIonly"]
# # dataset_mocks_list = ["DESY6_COLA_dec_below-23.5"]
# # dataset_mocks_list = ["DESY6_COLA_dec_above-23.5"]
# # dataset_mocks_list = ["DESY6_COLA", "DESY6_COLA_DR1tiles_noDESI", "DESY6_COLA_DR1tiles_DESIonly", "DESY6_COLA_dec_below-23.5", "DESY6_COLA_dec_above-23.5"]
# dynamical_theta_limits = False
# old_template = "_old"
# # old_template = ""
# delta_z = None # only for DESI stuff
# recon = "" # only for DESI stuff
# pow_broadband = [-2, -1, 0]

# Option B. DESI-related stuff
delta_z = 0.02
recon = "_recon"
suffix_list = ["_LRG1", "_LRG2", "_LRG3"]
dataset_mocks_list = [
    f"DESIY1_LRG_Abacus_altmtl{recon}_deltaz{delta_z}{suffix}"
    for suffix in suffix_list
]
dynamical_theta_limits = True
pow_broadband = [-1, 0]

all_samples = {}

for dataset in dataset_mocks_list:

    # Default
    dataset_orig = dataset
    bins_removed = []

    all_samples[dataset] = {}

    if delta_z is not None:
        if delta_z == 0.028:
            if "DESIY1" in dataset:
                if "LRG1" in dataset:
                    bins_removed = list(map(int, np.arange(7, 25)))  # LRG1
                    dataset_orig = dataset.replace("_LRG1", "")
                elif "LRG2" in dataset:
                    bins_removed = list(map(int, np.concatenate([np.arange(0, 7), np.arange(14, 25)])))  # LRG2
                    dataset_orig = dataset.replace("_LRG2", "")
                elif "LRG3" in dataset:
                    bins_removed = list(map(int, np.arange(0, 14)))  # LRG3
                    dataset_orig = dataset.replace("_LRG3", "")
        elif delta_z == 0.02:
            if "DESIY1" in dataset:
                if "LRG1" in dataset:
                    bins_removed = list(map(int, np.arange(10, 35)))  # LRG1
                    dataset_orig = dataset.replace("_LRG1", "")
                elif "LRG2" in dataset:
                    bins_removed = list(map(int, np.concatenate([np.arange(0, 10), np.arange(20, 35)])))  # LRG2
                    dataset_orig = dataset.replace("_LRG2", "")
                elif "LRG3" in dataset:
                    bins_removed = list(map(int, np.arange(0, 20)))  # LRG3
                    dataset_orig = dataset.replace("_LRG3", "")

    print(f"Loading the results for the {dataset} data set...")

    if "DESIY1" in dataset:
        nz_flag = "mocks"
        cov_type = "mocks"
        cosmology_template = "desifid"
        cosmology_covariance = "desifid"
    
        if "EZ" in dataset:
            n_mocks = 1000
        elif "Abacus" in dataset:
            n_mocks = 25
        
    elif "DESY6" in dataset:
        nz_flag = "mocks"
        cov_type = "cosmolike"
        # cov_type = "mocks"
        cosmology_template = "mice" + old_template
        cosmology_covariance = "mice"
        
        n_mocks = 1952
        
    theta_min, theta_max = GetThetaLimits(dataset=dataset_orig, nz_flag=nz_flag, dynamical_theta_limits=dynamical_theta_limits, code_path=code_path).get_theta_limits()

    # mock_id_list = ["mean"] + list(range(n_mocks))
    mock_id_list = list(range(n_mocks))

    # mock_id_list = [1]

    for mock_id in mock_id_list:
    
        # 1. Arguments
        base_config = {
            "dataset": dataset_orig,
            "weight_type": 1, # it will not be used...
            "mock_id": mock_id,
            "nz_flag": nz_flag,
            "cov_type": cov_type,
            "cosmology_template": cosmology_template,
            "cosmology_covariance": cosmology_covariance,
            "delta_theta": 0.2,
            "theta_min": theta_min,
            "theta_max": theta_max,
            "pow_broadband": pow_broadband,
            "bins_removed": bins_removed,
            "alpha_min": 0.8,
            "alpha_max": 1.2,
            # "alpha_type": "alpha_wigg_only",
            "alpha_type": "alpha_wigg_nowigg",
            "code_path": code_path,
            "save_path": save_path,
        }
        
        baofit_checker = BAOFitChecker(**base_config)
    
        # 2. Run cosmological inference
        inference = BAOInference(
            baofit_checker=baofit_checker,
            nwalkers=32,
            nsteps=5000,
            burnin=1000,
            # overwrite=True,
            verbose=True
        )

        if not inference.valid:
            continue
    
        # MCMC starting point
        initial_guess = [100.0, 0.3]  # h*r_d, Omega_m
        samples, log_probs = inference.run_mcmc(initial_guess)

        # Summarize MCMC
        results = inference.summarize_chain(samples)

        # Minimizer for best-fit
        # best_fit = inference.run_minimizer(samples, log_probs)
        best_fit = None
    
        # Optionally make triangle plot
        inference.make_triangle_plot(samples, best_fit, close_fig=True)

        all_samples[dataset][mock_id] = samples

        print()
        

In [ ]:
from getdist import MCSamples, plots

bounds = [(60.0, 130.0), (0.1, 0.9)]

names = ["hrd", "Omega_m"]
labels = [r"h\,r_{\rm d}\ [{\rm Mpc}/h]", r"\Omega_{\rm m}"]

ranges = {
    "hrd": list(bounds[0]),
    "Omega_m": list(bounds[1]),
}

for mock_id in mock_id_list:

    print(f"Analyzing mock number {mock_id}")

    gdsamples_list = []
    legend_labels = []
    colors = []

    for dataset in dataset_mocks_list:

        # skip if dataset or mock_id missing
        if dataset not in all_samples:
            continue
        if mock_id not in all_samples[dataset]:
            continue

        samples = all_samples[dataset][mock_id]

        gdsamples = MCSamples(
            samples=samples,
            names=names,
            labels=labels,
            ranges=ranges,
        )

        gdsamples.updateSettings({
            "smooth_scale_2D": 0.3,
            "smooth_scale_1D": 0.3,
        })

        gdsamples_list.append(gdsamples)

        if "LRG1" in dataset:
            legend_labels.append("LRG1")
            colors.append("darkblue")
        elif "LRG2" in dataset:
            legend_labels.append("LRG2")
            colors.append("firebrick")
        elif "LRG3" in dataset:
            legend_labels.append("LRG3")
            colors.append("green")
        else:
            legend_labels.append(dataset)
            colors.append("black")

    # skip plot if nothing available
    if len(gdsamples_list) == 0:
        print(f"Mock {mock_id}: no valid datasets")
        continue

    g = plots.get_subplot_plotter()
    g.settings.axis_marker_ls = "--"
    g.settings.axis_marker_color = "black"

    fig = g.triangle_plot(
        gdsamples_list,
        filled=True,
        legend_labels=legend_labels,
        contour_colors=colors
    )

    # plt.savefig(f"triangle_mock{mock_id}_LRG123.png", bbox_inches="tight")
    # plt.close(fig)